In [0]:
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt

In [0]:
import os

# Read the CSV from the same folder as this notebook
_user = spark.sql("SELECT current_user()").first()[0]
csv_path = f"/Workspace/Users/{_user}/databricks_ai/DeepLearning/RSCCASN.csv"

df = pd.read_csv(csv_path,parse_dates=True,index_col='DATE')
test_size = 18
test_ind = len(df) - test_size
train = df.iloc[:test_ind]
test = df.iloc[test_ind:]

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train)
test_scaled = scaler.transform(test)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator



In [0]:
length = 12
generator = TimeseriesGenerator(train_scaled, train_scaled, length=length, batch_size=1)
n_features = 1

model = Sequential()
model.add(LSTM(100, activation='relu', input_shape=(length, n_features)))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')
model.summary()
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=2)

validation_generator = TimeseriesGenerator(test_scaled, test_scaled, length=length, batch_size=1)
model.fit(generator, epochs=20,
          validation_data=validation_generator,
          callbacks=[early_stop])
loss_df = pd.DataFrame(model.history.history)
loss_df.plot()

In [0]:
test_predictions = []
first_eval_batch = train_scaled[-length:]
current_batch = first_eval_batch.reshape((1, length, n_features))

for i in range(len(test)):
    current_pred = model.predict(current_batch)[0]
    test_predictions.append(current_pred)
    current_batch = np.append(current_batch[:,1:,:],[[current_pred]],axis=1)
true_predictions = scaler.inverse_transform(test_predictions)
test['Predictions'] = true_predictions
test.plot(figsize=(12,8))

In [0]:
full_scaler = MinMaxScaler(feature_range=(0, 1))
scaled_full_data = full_scaler.fit_transform(df)
generator = TimeseriesGenerator(scaled_full_data, scaled_full_data, length=length, batch_size=1)
model = Sequential()
model.add(LSTM(100, activation='relu', input_shape=(length, n_features)))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')
model.summary()

model.fit(generator, epochs=8)

In [0]:
forecast = []
periods = 12

first_eval_batch = scaled_full_data[-length:]
current_batch = first_eval_batch.reshape((1, length, n_features))

for i in range(periods):
    current_pred = model.predict(current_batch)[0]
    forecast.append(current_pred)
    current_batch = np.append(current_batch[:,1:,:],[[current_pred]],axis=1)
forecast = scaler.inverse_transform(forecast)
forecast_index = pd.date_range(start=df.index[-1], periods=periods, freq="MS")
forecast_df = pd.DataFrame(data=forecast, index=forecast_index, columns=['Forecast'])

ax = df.plot()
forecast_df.plot(ax=ax)
